In [3]:
# Anbindung LLM Christian

In [37]:
import json
from openai import OpenAI
import requests
import httpx

with open("TS-Scanner.json") as f:
    env = json.load(f)

with open("llm.codebar.net.json") as f:
    codebar = json.load(f)

access = {v["name"]: v["value"] for v in env["variables"] if v.get("enabled", True)}


class ScrubTransport(httpx.HTTPTransport):
    def handle_request(self, request):
        request.headers["user-agent"] = "Mozilla/5.0"
        for name in [h for h in request.headers if h.lower().startswith("x-stainless")]:
            del request.headers[name]
        return super().handle_request(request)

client = OpenAI(
    base_url=access["url"],
    api_key=access["token"],
    http_client=httpx.Client(transport=ScrubTransport()),
)

models = client.models.list()
print([m.id for m in models.data])


['qwen2.5:7b-instruct', 'qwen2.5:32b-instruct-q4_K_M', 'qwen2.5-coder:32b', 'llama3.3:70b-instruct-q4_K_M', 'qwen3:32b', 'qwen3:30b-a3b', 'qwen3.5:4b', 'qwen3.5:9b', 'qwen3.6:35b', 'deepseek-r1:70b', 'gpt-oss:20b']


In [38]:
import requests

url = access['url'].rstrip('/') + '/models'   # adjust /v1 if needed
headers = {"Authorization": f"Bearer {access['token']}"}

r = requests.get(url, headers=headers)
print("Status:", r.status_code)
print("Headers:", dict(r.headers))
print("Body:", r.text[:1000])

Status: 200
Headers: {'Date': 'Thu, 28 May 2026 19:40:28 GMT', 'Content-Type': 'application/json', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Server': 'cloudflare', 'cf-cache-status': 'DYNAMIC', 'Nel': '{"report_to":"cf-nel","success_fraction":0.0,"max_age":604800}', 'Report-To': '{"group":"cf-nel","max_age":604800,"endpoints":[{"url":"https://a.nel.cloudflare.com/report/v4?s=%2F4XQ0wfMcPJtXxn4lQTw%2FKZZU4py8bThbgW3LrVIniwnnZMu0Ik1UA7h2xr4Zrhu6t6OCj2NrNeawGyTdf%2B1zOs%2F44VgKhTL2ZIDWFPTWZahvrX1Dq7RBuVkP3lt7Q7nhO8%3D"}]}', 'Content-Encoding': 'gzip', 'CF-RAY': 'a02fbb33be14bbf6-ZRH', 'alt-svc': 'h3=":443"; ma=86400'}
Body: {"data":[{"id":"qwen2.5:7b-instruct","object":"model","created":1677610602,"owned_by":"openai"},{"id":"qwen2.5:32b-instruct-q4_K_M","object":"model","created":1677610602,"owned_by":"openai"},{"id":"qwen2.5-coder:32b","object":"model","created":1677610602,"owned_by":"openai"},{"id":"llama3.3:70b-instruct-q4_K_M","object":"model","created":1677610602,"

In [39]:
def chat(prompt, model="qwen2.5:32b-instruct-q4_K_M", system=None, stream=False, **kwargs):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    if stream:
        text = ""
        for chunk in client.chat.completions.create(
            model=model, messages=messages, stream=True, **kwargs
        ):
            delta = chunk.choices[0].delta.content or ""
            print(delta, end="", flush=True)
            text += delta
        print()
        return text

    resp = client.chat.completions.create(model=model, messages=messages, **kwargs)
    return resp.choices[0].message.content

## Test LLM

In [40]:
print(chat("Was ist eine Tierseuche? Erkläre in 100 Zeichen"))

Eine Tierseuche ist eine ansteckende Krankheit bei Tieren, die sich schnell ausbreitet und viele Tiere betroffen macht.


In [43]:
!python interpreter.py -i ../../../data/unstructured/gefluegelnews/disease_reports.jsonl  -o ../../../data/unstructured/gefluegelnews/disease_reports_embeddings.jsonl --progress-every 100

['qwen2.5:7b-instruct', 'qwen2.5:32b-instruct-q4_K_M', 'qwen2.5-coder:32b', 'llama3.3:70b-instruct-q4_K_M', 'qwen3:32b', 'qwen3:30b-a3b', 'qwen3.5:4b', 'qwen3.5:9b', 'qwen3.6:35b', 'deepseek-r1:70b', 'gpt-oss:20b']
resuming after 4 already-processed lines
done


In [45]:
import json
import pandas as pd

records = [json.loads(l) for l in open("../../../data/unstructured/gefluegelnews/disease_reports_embeddings.jsonl", encoding="utf-8")]
df = pd.json_normalize(records)   # flattens consequence.politisch etc.

cols = ["Disease", "DiseaseSubtype", "InEurope","Fulltext"
        "consequence.politisch", "consequence.sozial", "consequence.wirtschaftlich",
        "_error"]

df[[c for c in cols if c in df.columns]]

,_error
0,extraction: BadRequestError: Error code: 400 -...
1,extraction: BadRequestError: Error code: 400 -...
2,extraction: BadRequestError: Error code: 400 -...
3,extraction: BadRequestError: Error code: 400 -...


In [46]:
!python interpreter.py \
  -s SystemPrompt.md \
  -i ../../../data/unstructured/PAFF_extracted \
  -o ../../../data/unstructured/PAFF_embedded \
  --progress-every 100

['qwen2.5:7b-instruct', 'qwen2.5:32b-instruct-q4_K_M', 'qwen2.5-coder:32b', 'llama3.3:70b-instruct-q4_K_M', 'qwen3:32b', 'qwen3:30b-a3b', 'qwen3.5:4b', 'qwen3.5:9b', 'qwen3.6:35b', 'deepseek-r1:70b', 'gpt-oss:20b']
found 15 json files in ../../../data/unstructured/PAFF_extracted
done
